# Uruguay Labor Market (2004–2023): A Data Science Project

**Domain:** Economics — Labor Market

**Objective:** Build an end-to-end, reproducible data science project that explores Uruguay's labor market over the last 20 years using public data. We will:

- Ask initial questions to frame the analysis.
- Collect data from the World Bank API.
- Perform exploratory data analysis (EDA).
- Build regression and time-series models.
- Interpret results and document insights.

---

## Initial Questions (Problem Framing)

1. How has Uruguay's unemployment rate evolved over the last 20 years?
2. Do labor force participation and employment rates move together with unemployment?
3. Can we model unemployment using macro labor indicators?
4. Can a time-series model capture trends and forecast short-term unemployment?

## 1. Data Collection

We will use the **World Bank API**, which provides clean, public indicators for most countries.

Indicators used:
- **Unemployment, total (% of total labor force)** — `SL.UEM.TOTL.ZS`
- **Labor force participation rate, total (% ages 15+)** — `SL.TLF.CACT.ZS`
- **Employment to population ratio, 15+, total (%)** — `SL.EMP.TOTL.SP.ZS`

Country:
- **Uruguay** (`URY`)

Time period:
- **2004–2023** (latest 20 years)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm

sns.set_theme(style="whitegrid")

In [ ]:
def fetch_world_bank_indicator(country_code: str, indicator: str, start: int, end: int) -> pd.DataFrame:
    url = (
        f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator}"
        f"?format=json&per_page=200&date={start}:{end}"
    )
    data = pd.read_json(url)
    records = data[1]
    df = pd.DataFrame(records)
    df = df[["date", "value"]].rename(columns={"date": "year", "value": indicator})
    df["year"] = df["year"].astype(int)
    return df

country = "URY"
start_year, end_year = 2004, 2023

indicators = {
    "SL.UEM.TOTL.ZS": "unemployment_rate",
    "SL.TLF.CACT.ZS": "labor_force_participation",
    "SL.EMP.TOTL.SP.ZS": "employment_to_population",
}

frames = []
for indicator_code, name in indicators.items():
    df = fetch_world_bank_indicator(country, indicator_code, start_year, end_year)
    df = df.rename(columns={indicator_code: name})
    frames.append(df)

data = frames[0]
for frame in frames[1:]:
    data = data.merge(frame, on="year", how="outer")

data = data.sort_values("year").reset_index(drop=True)

data

## 2. Data Cleaning

We will check missing values and keep only years with complete observations.

In [ ]:
data.isna().sum()

In [ ]:
clean_data = data.dropna().reset_index(drop=True)
clean_data

## 3. Exploratory Data Analysis (EDA)

We visualize the trends for each indicator.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(clean_data["year"], clean_data["unemployment_rate"], marker="o")
plt.title("Uruguay Unemployment Rate (2004–2023)")
plt.xlabel("Year")
plt.ylabel("Unemployment Rate (% of labor force)")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=clean_data, x="year", y="labor_force_participation", marker="o", ax=ax[0])
ax[0].set_title("Labor Force Participation Rate")
ax[0].set_xlabel("Year")
ax[0].set_ylabel("% ages 15+")

sns.lineplot(data=clean_data, x="year", y="employment_to_population", marker="o", ax=ax[1])
ax[1].set_title("Employment to Population Ratio")
ax[1].set_xlabel("Year")
ax[1].set_ylabel("% ages 15+")

plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(clean_data.drop(columns=["year"]))
plt.show()

## 4. Regression Model

We model unemployment as a function of labor force participation and employment-to-population ratio.

This is a simple linear regression to help interpret relationships.

In [ ]:
X = clean_data[["labor_force_participation", "employment_to_population"]]
y = clean_data["unemployment_rate"]

model = LinearRegression()
model.fit(X, y)

coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_
})

intercept = model.intercept_
coefficients, intercept

In [ ]:
predictions = model.predict(X)

mae = mean_absolute_error(y, predictions)
rmse = mean_squared_error(y, predictions, squared=False)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, predictions)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")
plt.xlabel("Actual Unemployment Rate")
plt.ylabel("Predicted Unemployment Rate")
plt.title("Regression Fit")
plt.show()

## 5. Time-Series Modeling

We use a basic ARIMA model to capture the trend in unemployment rates and produce a short-term forecast.

In [ ]:
ts = clean_data.set_index("year")["unemployment_rate"]

# Fit ARIMA(1,1,1) as a simple baseline
model_arima = sm.tsa.ARIMA(ts, order=(1, 1, 1))
arima_result = model_arima.fit()

arima_result.summary()

In [ ]:
forecast_steps = 3
forecast = arima_result.forecast(steps=forecast_steps)

forecast_years = list(range(ts.index.max() + 1, ts.index.max() + 1 + forecast_steps))

plt.figure(figsize=(8, 4))
plt.plot(ts.index, ts.values, label="Historical")
plt.plot(forecast_years, forecast.values, label="Forecast", marker="o")
plt.title("Unemployment Rate Forecast")
plt.xlabel("Year")
plt.ylabel("Unemployment Rate (%)")
plt.legend()
plt.show()

## 6. Conclusions

**Key takeaways:**

- Uruguay's unemployment shows cycles but a manageable long-term trend.
- Labor force participation and employment-to-population help explain unemployment variations.
- A linear regression offers interpretable coefficients.
- A simple ARIMA model provides a baseline forecast.

**Next steps:**

- Add additional macroeconomic variables (GDP growth, inflation).
- Test more robust time-series models (SARIMA, Prophet).
- Evaluate out-of-sample performance using time-series cross-validation.